## Load cleaned settlement dataset and growth dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

main_lakes = pd.read_csv(
    '../data/processed/glofguard_lakes_v2_settlement_clean.csv'
)

growth_data = pd.read_csv(
    '../data/processed/cpec_growth_rate_clean.csv'
)

print("Main dataset:", main_lakes.shape)
print("Growth dataset:", growth_data.shape)

Main dataset: (8806, 11)
Growth dataset: (1864, 7)


### Match historical growth data to main lake inventory

In [2]:
main_coords = np.radians(
    main_lakes[['latitude', 'longitude']].values
)

growth_coords = np.radians(
    growth_data[['latitude', 'longitude']].values
)

growth_tree = BallTree(
    growth_coords,
    metric='haversine'
)

distances, indices = growth_tree.query(
    main_coords,
    k=1
)

growth_distance_km = (
    distances.flatten() * 6371.0
)

GROWTH_MATCH_THRESHOLD_KM = 1.0

has_growth = (
    growth_distance_km <= GROWTH_MATCH_THRESHOLD_KM
)

print("Total lakes:", len(main_lakes))
print("Reliable growth matches:", has_growth.sum())
print("Missing growth:", (~has_growth).sum())
print("Coverage:", has_growth.mean() * 100)

Total lakes: 8806
Reliable growth matches: 3195
Missing growth: 5611
Coverage: 36.282080399727455


### Add historical growth feature

In [3]:
main_lakes['annual_area_change_km2_per_year'] = np.nan

main_lakes.loc[
    has_growth,
    'annual_area_change_km2_per_year'
] = (
    growth_data.iloc[
        indices[has_growth].flatten()
    ]['annual_area_change_km2_per_year'].values
)

# Indicator showing whether historical growth data exists
main_lakes['growth_data_available'] = (
    main_lakes['annual_area_change_km2_per_year']
    .notna()
    .astype(int)
)

print(
    main_lakes[
        [
            'annual_area_change_km2_per_year',
            'growth_data_available'
        ]
    ].head(10)
)

   annual_area_change_km2_per_year  growth_data_available
0                              NaN                      0
1                              NaN                      0
2                              NaN                      0
3                              NaN                      0
4                              NaN                      0
5                              NaN                      0
6                              NaN                      0
7                              NaN                      0
8                              NaN                      0
9                              NaN                      0


### Final dataset quality checks

In [4]:
print("Final shape:", main_lakes.shape)

print("\nColumns:")
print(main_lakes.columns.tolist())

print("\nMissing values:")
print(main_lakes.isna().sum())

print("\nDuplicate rows:")
print(main_lakes.duplicated().sum())

print("\nGrowth availability:")
print(
    main_lakes['growth_data_available']
    .value_counts()
)

print("\nGrowth feature summary:")
print(
    main_lakes[
        ['annual_area_change_km2_per_year']
    ].describe()
)

Final shape: (8806, 13)

Columns:
['sample_id', 'area', 'longitude', 'latitude', 'is_top200', 'temperature', 'rainfall', 'label', 'elevation', 'distance_to_nearest_settlement_km', 'nearest_settlement_name', 'annual_area_change_km2_per_year', 'growth_data_available']

Missing values:
sample_id                               0
area                                    0
longitude                               0
latitude                                0
is_top200                               0
temperature                             0
rainfall                                0
label                                   0
elevation                               0
distance_to_nearest_settlement_km       0
nearest_settlement_name                 0
annual_area_change_km2_per_year      5611
growth_data_available                   0
dtype: int64

Duplicate rows:
0

Growth availability:
growth_data_available
0    5611
1    3195
Name: count, dtype: int64

Growth feature summary:
       annual_area_chan

### Save final dataset — glofguard_training_data.csv

In [5]:
final_path = (
    '../data/final/glofguard_training_data.csv'
)

main_lakes.to_csv(
    final_path,
    index=False
)

print("Saved:", final_path)
print("Final shape:", main_lakes.shape)

Saved: ../data/final/glofguard_training_data.csv
Final shape: (8806, 13)
